In [1]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from tqdm.auto import tqdm
from google.oauth2 import service_account
import numpy as np

# 1. 신분증(JSON 키) 경로 지정
KEY_PATH = '../google_key.json'

# 2. 인증 객체 생성
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# 3. 프로젝트 ID 설정
project_id = 'gdelt-analysis-494301'

# 4. 데이터 불러오기
query = "SELECT SQLDATE FROM `gdelt-bq.full.events` LIMIT 5"
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("인증 성공! 데이터를 가져왔습니다.")

Downloading: 100%|██████████|
인증 성공! 데이터를 가져왔습니다.


In [2]:
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE,
    EventCode,
    QuadClass,
    GoldsteinScale,
    NumMentions,
    NumArticles,
    AvgTone,
    Actor1CountryCode,
    Actor2CountryCode,
    Actor1Type1Code,
    Actor2Type1Code,
    Actor1Geo_Type,
    Actor1Geo_Lat,
    Actor1Geo_Long,
    Actor2Geo_Type,
    Actor2Geo_Lat,
    Actor2Geo_Long,
    ActionGeo_Type,
    ActionGeo_Lat,
    ActionGeo_Long,
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1
  AND ActionGeo_Type IN (3, 4, 5)
"""

# 2. 데이터 불러오기
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("데이터 불러오기 완료")
display(df.head())

Downloading: 100%|██████████|
데이터 불러오기 완료


,SQLDATE,EventCode,QuadClass,GoldsteinScale,NumMentions,NumArticles,AvgTone,Actor1CountryCode,Actor2CountryCode,Actor1Type1Code,...,Actor1Geo_Type,Actor1Geo_Lat,Actor1Geo_Long,Actor2Geo_Type,Actor2Geo_Lat,Actor2Geo_Long,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20160522,150,4,-7.2,5,5,-0.806452,CHN,TWN,None,...,4,39.9289,116.388,4,25.0327,121.275,4,39.9289,116.388,http://bilbaoya.com/2016/05/23/taiwan-swears-i...
1,20160522,150,4,-7.2,5,5,0.634249,CHN,TWN,None,...,4,25.0327,121.275,4,25.0327,121.275,4,25.0327,121.275,http://fait-religieux.com/2016/05/22/taiwan-sw...
2,20260512,190,4,-10.0,25,15,-1.751929,TWN,CHN,None,...,4,24.0000,119.000,4,39.9289,116.388,4,39.9289,116.388,https://www.yahoo.com/news/articles/key-events...
3,20260512,194,4,-10.0,14,11,-1.751929,TWN,CHN,None,...,4,24.0000,119.000,4,39.9289,116.388,4,39.9289,116.388,https://www.yahoo.com/news/articles/key-events...
4,20260512,192,4,-9.5,4,4,-1.831123,CHN,TWN,None,...,4,25.0478,121.532,4,39.9289,116.388,4,39.9289,116.388,https://www.hindustantimes.com/world-news/ahea...


In [3]:
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11941 entries, 0 to 11940
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   SQLDATE            11941 non-null  datetime64[ns]
 1   EventCode          11941 non-null  object        
 2   QuadClass          11941 non-null  Int64         
 3   GoldsteinScale     11941 non-null  float64       
 4   NumMentions        11941 non-null  Int64         
 5   NumArticles        11941 non-null  Int64         
 6   AvgTone            11941 non-null  float64       
 7   Actor1CountryCode  11941 non-null  object        
 8   Actor2CountryCode  11941 non-null  object        
 9   Actor1Type1Code    1995 non-null   object        
 10  Actor2Type1Code    1279 non-null   object        
 11  Actor1Geo_Type     11941 non-null  Int64         
 12  Actor1Geo_Lat      11941 non-null  float64       
 13  Actor1Geo_Long     11941 non-null  float64       
 14  Actor2

In [5]:
# CHN인지 TWN인지 비율
print(df['Actor1CountryCode'].value_counts())
print(df['Actor2CountryCode'].value_counts())

Actor1CountryCode
CHN    7342
TWN    4599
Name: count, dtype: int64
Actor2CountryCode
TWN    7342
CHN    4599
Name: count, dtype: int64


같은 이벤트가 양쪽으로 기록되었을 가능성 있음...

In [6]:
# 중복 여부 확인 필요
# 같은 날짜 + 같은 EventCode + 같은 ActionGeo → 중복 가능성
df.duplicated(subset=['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long']).sum()

np.int64(5051)

In [7]:
# 결측이 아닌 경우 어떤 값들이 있는지 확인
print(df['Actor1Type1Code'].value_counts())
print(df['Actor2Type1Code'].value_counts())

Actor1Type1Code
MIL    1330
GOV     504
CVL      35
MED      19
BUS      17
UAF      12
ELI      11
LEG      11
SEP      11
OPP      10
MNC       9
COP       5
LAB       5
JUD       3
EDU       3
NGO       2
SPY       2
REB       2
HLH       2
AGR       1
CRM       1
Name: count, dtype: int64
Actor2Type1Code
MIL    788
GOV    270
CVL     79
LAB     31
LEG     19
UAF     17
BUS     14
OPP     14
EDU     11
MED      9
ELI      8
SEP      6
JUD      5
COP      3
AGR      2
REB      2
HLH      1
Name: count, dtype: int64


In [8]:
# ActionGeo가 Actor1(중국) 쪽인지 Actor2(대만) 쪽인지
chn_to_twn = df[df['Actor1CountryCode'] == 'CHN']

is_actor1 = (
    (chn_to_twn['ActionGeo_Lat'] == chn_to_twn['Actor1Geo_Lat']) &
    (chn_to_twn['ActionGeo_Long'] == chn_to_twn['Actor1Geo_Long'])
)
is_actor2 = (
    (chn_to_twn['ActionGeo_Lat'] == chn_to_twn['Actor2Geo_Lat']) &
    (chn_to_twn['ActionGeo_Long'] == chn_to_twn['Actor2Geo_Long'])
)

print(f"Actor1만:        {(is_actor1 & ~is_actor2).sum()}")
print(f"Actor2만:        {(~is_actor1 & is_actor2).sum()}")
print(f"둘 다 일치:      {(is_actor1 & is_actor2).sum()}")  # ← 이게 있을 것
print(f"둘 다 아님:      {(~is_actor1 & ~is_actor2).sum()}")

Actor1만:        2075
Actor2만:        2178
둘 다 일치:      2386
둘 다 아님:      703


In [9]:
# 중복 의심 그룹 확인 (제거하지 않고 보기만)
dup_mask = df.duplicated(subset=[
    'SQLDATE', 'EventCode', 
    'ActionGeo_Lat', 'ActionGeo_Long'
], keep=False)

df_dup = df[dup_mask].sort_values(['SQLDATE', 'EventCode', 'ActionGeo_Lat'])

# 실제로 어떻게 생겼는지 확인
df_dup[['SQLDATE', 'EventCode', 
        'Actor1CountryCode', 'Actor2CountryCode',
        'ActionGeo_Lat', 'ActionGeo_Long', 
        'SOURCEURL']].head(20)

,SQLDATE,EventCode,Actor1CountryCode,Actor2CountryCode,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
3695,2013-04-14,190,TWN,CHN,23.5919,108.373,http://www.themalaysianinsider.com/malaysia/ar...
5415,2013-04-14,190,TWN,CHN,23.5919,108.373,http://www.themalaysianinsider.com/malaysia/ar...
11575,2013-04-15,190,TWN,CHN,23.5919,108.373,http://blog.limkitsiang.com/2014/04/15/kit-sia...
11768,2013-04-15,190,TWN,CHN,23.5919,108.373,http://blog.limkitsiang.com/2014/04/15/kit-sia...
2430,2013-09-18,190,CHN,TWN,24.0000,119.000,http://timesofindia.indiatimes.com/world/china...
5669,2013-09-18,190,CHN,TWN,24.0000,119.000,http://www.indianexpress.com/news/21-held-for-...
1800,2013-09-25,194,TWN,CHN,22.7902,115.239,http://www.theborneopost.com/2013/09/27/taiwan...
4691,2013-09-25,194,TWN,CHN,22.7902,115.239,http://thepeninsulaqatar.com/asia/254742-taiwa...
5617,2013-09-25,194,TWN,CHN,22.7902,115.239,http://main.omanobserver.om/?p=16731
7225,2013-09-26,194,TWN,CHN,22.7902,115.239,http://www.scmp.com/news/china/article/1318338...


In [10]:
# URL 전체 출력
pd.set_option('display.max_colwidth', None)

df_dup.sort_values('SQLDATE', ascending=False).head(10)

,SQLDATE,EventCode,QuadClass,GoldsteinScale,NumMentions,NumArticles,AvgTone,Actor1CountryCode,Actor2CountryCode,Actor1Type1Code,...,Actor1Geo_Type,Actor1Geo_Lat,Actor1Geo_Long,Actor2Geo_Type,Actor2Geo_Lat,Actor2Geo_Long,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
32,2026-05-12,194,4,-10.0,1,1,-1.795580,TWN,CHN,None,...,4,24.0000,119.000,4,24.4367,118.318,4,39.9289,116.388,https://thelinkpaper.ca/ahead-of-trumps-beijing-visit-a-look-at-ties-between-us-china-and-taiwan/
7,2026-05-12,190,4,-10.0,6,6,-1.728723,TWN,CHN,None,...,4,24.4367,118.318,4,39.9289,116.388,4,39.9289,116.388,https://www.thehindu.com/news/international/key-events-in-ties-between-the-united-states-china-and-taiwan/article70968350.ece
18,2026-05-12,192,4,-9.5,34,34,-1.774812,CHN,TWN,None,...,4,39.9289,116.388,4,25.0478,121.532,4,39.9289,116.388,https://www.yahoo.com/news/articles/key-events-ties-between-united-031852751.html
4,2026-05-12,192,4,-9.5,4,4,-1.831123,CHN,TWN,None,...,4,25.0478,121.532,4,39.9289,116.388,4,39.9289,116.388,https://www.hindustantimes.com/world-news/ahead-of-trumps-beijing-visit-a-look-at-ties-between-us-china-and-taiwan-101778560156964.html
30,2026-05-12,190,4,-10.0,3,3,-1.795580,CHN,TWN,None,...,4,39.9289,116.388,4,24.4367,118.318,4,39.9289,116.388,https://thelinkpaper.ca/ahead-of-trumps-beijing-visit-a-look-at-ties-between-us-china-and-taiwan/
24,2026-05-12,190,4,-10.0,5,5,-2.483611,CHN,TWN,None,...,4,39.9289,116.388,4,25.0478,121.532,4,39.9289,116.388,https://thelinkpaper.ca/ahead-of-trumps-beijing-visit-a-look-at-ties-between-us-china-and-taiwan/
20,2026-05-12,190,4,-10.0,2,2,-1.708279,TWN,CHN,None,...,4,24.0000,119.000,4,25.0478,121.532,4,39.9289,116.388,https://www.yahoo.com/news/articles/key-events-ties-between-united-031852751.html
19,2026-05-12,190,4,-10.0,3,3,-1.708279,TWN,CHN,None,...,4,39.9289,116.388,4,25.0478,121.532,4,39.9289,116.388,https://www.yahoo.com/news/articles/key-events-ties-between-united-031852751.html
5,2026-05-12,190,4,-10.0,26,16,-1.797695,TWN,CHN,None,...,4,25.0478,121.532,4,39.9289,116.388,4,39.9289,116.388,https://www.hindustantimes.com/world-news/ahead-of-trumps-beijing-visit-a-look-at-ties-between-us-china-and-taiwan-101778560156964.html
36,2026-05-12,194,4,-10.0,1,1,-1.795580,CHN,TWN,None,...,4,39.9289,116.388,4,24.4367,118.318,4,24.0000,119.000,https://thelinkpaper.ca/ahead-of-trumps-beijing-visit-a-look-at-ties-between-us-china-and-taiwan/


In [13]:
mil_df = df[(df['Actor1Type1Code'] == 'MIL') | (df['Actor2Type1Code'] == 'MIL')].copy()

print(f"전체 데이터 수: {len(df)}")
print(f"MIL 필터링 후 데이터 수: {len(mil_df)}")

전체 데이터 수: 11941
MIL 필터링 후 데이터 수: 2049


In [14]:
# 지정된 컬럼 조합이 중복된 행의 총 개수 (첫 번째 행 제외)
duplicate_count = mil_df.duplicated(subset=['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long']).sum()

print(f"중복된 데이터 건수: {duplicate_count}건")
print(f"전체 데이터 대비 중복 비율: {(duplicate_count / len(mil_df)) * 100:.2f}%")

중복된 데이터 건수: 543건
전체 데이터 대비 중복 비율: 26.50%


In [15]:
mil_df.sort_values('SQLDATE', ascending=False).head(10)

,SQLDATE,EventCode,QuadClass,GoldsteinScale,NumMentions,NumArticles,AvgTone,Actor1CountryCode,Actor2CountryCode,Actor1Type1Code,...,Actor1Geo_Type,Actor1Geo_Lat,Actor1Geo_Long,Actor2Geo_Type,Actor2Geo_Lat,Actor2Geo_Long,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
3294,2026-04-30,150,4,-7.2,9,9,0.559158,CHN,TWN,MIL,...,4,39.9289,116.388,4,25.0478,121.532,4,39.9289,116.388,https://focustaiwan.tw/politics/202604300010
4154,2026-04-29,150,4,-7.2,10,10,-7.010429,CHN,TWN,MIL,...,4,39.9289,116.388,4,25.0478,121.532,4,39.9289,116.388,https://foreignpolicy.com/2026/04/29/taiwan-china-us-war-invasion-crisis-evacuation-civilians/
4840,2026-04-28,152,4,-7.2,4,4,-4.794521,TWN,CHN,GOV,...,4,25.0478,121.532,4,24.5603,117.547,4,24.5603,117.547,https://www.khaama.com/taiwan-puts-forces-on-alert-after-detecting-chinese-warships-near-penghu/
4839,2026-04-28,152,4,-7.2,2,2,-4.794521,TWN,CHN,GOV,...,4,25.0478,121.532,4,24.5603,117.547,4,25.0478,121.532,https://www.khaama.com/taiwan-puts-forces-on-alert-after-detecting-chinese-warships-near-penghu/
4420,2026-04-24,150,4,-7.2,10,10,-4.290429,CHN,TWN,MIL,...,4,28.0355,119.794,4,28.0355,119.794,4,28.0355,119.794,https://maritime-executive.com/article/china-s-naval-drills-show-growing-focus-on-capturing-taiwan
8215,2026-04-16,150,4,-7.2,2,2,0.397614,TWN,CHN,None,...,4,24.1098,113.005,4,24.1098,113.005,4,24.1098,113.005,https://www.taipeitimes.com/News/taiwan/archives/2026/04/17/2003855757
8214,2026-04-15,150,4,-7.2,4,4,0.397614,TWN,CHN,None,...,4,24.1098,113.005,4,24.1098,113.005,4,24.1098,113.005,https://www.taipeitimes.com/News/taiwan/archives/2026/04/17/2003855757
5836,2026-03-16,190,4,-10.0,2,2,-0.608519,TWN,CHN,None,...,4,46.3167,129.567,4,46.3167,129.567,4,46.3167,129.567,https://www.navalnews.com/naval-news/2026/03/taiwan-navy-takes-delivery-of-the-first-tuo-chiang-class-batch-2-catamaran-corvette/
5837,2026-03-16,190,4,-10.0,2,2,-0.608519,TWN,CHN,None,...,4,23.5953,119.607,4,46.3167,129.567,4,23.5953,119.607,https://www.navalnews.com/naval-news/2026/03/taiwan-navy-takes-delivery-of-the-first-tuo-chiang-class-batch-2-catamaran-corvette/
5838,2026-03-16,190,4,-10.0,6,6,-0.608519,TWN,CHN,MED,...,4,23.5953,119.607,4,46.3167,129.567,4,23.5953,119.607,https://www.navalnews.com/naval-news/2026/03/taiwan-navy-takes-delivery-of-the-first-tuo-chiang-class-batch-2-catamaran-corvette/


In [17]:
# 중복된 항목들만 따로 추출하여 상위 10개 확인
duplicates = mil_df[mil_df.duplicated(subset=['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long'], keep=False)]

# 확인을 위해 정렬하여 출력
duplicates.sort_values(by=['SQLDATE', 'ActionGeo_Lat'], ascending=[False,False]).head(10)

,SQLDATE,EventCode,QuadClass,GoldsteinScale,NumMentions,NumArticles,AvgTone,Actor1CountryCode,Actor2CountryCode,Actor1Type1Code,...,Actor1Geo_Type,Actor1Geo_Lat,Actor1Geo_Long,Actor2Geo_Type,Actor2Geo_Lat,Actor2Geo_Long,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
5837,2026-03-16,190,4,-10.0,2,2,-0.608519,TWN,CHN,None,...,4,23.5953,119.6070,4,46.3167,129.5670,4,23.5953,119.6070,https://www.navalnews.com/naval-news/2026/03/taiwan-navy-takes-delivery-of-the-first-tuo-chiang-class-batch-2-catamaran-corvette/
5838,2026-03-16,190,4,-10.0,6,6,-0.608519,TWN,CHN,MED,...,4,23.5953,119.6070,4,46.3167,129.5670,4,23.5953,119.6070,https://www.navalnews.com/naval-news/2026/03/taiwan-navy-takes-delivery-of-the-first-tuo-chiang-class-batch-2-catamaran-corvette/
4291,2026-01-20,190,4,-10.0,2,2,-7.929825,CHN,TWN,None,...,4,39.9289,116.3880,4,39.9289,116.3880,4,25.0478,121.5320,https://foreignpolicy.com/2026/01/19/china-taiwan-invasion-failed-xi-disaster/
4293,2026-01-20,190,4,-10.0,6,6,-7.929825,CHN,TWN,None,...,4,39.9289,116.3880,4,25.0478,121.5320,4,25.0478,121.5320,https://foreignpolicy.com/2026/01/19/china-taiwan-invasion-failed-xi-disaster/
3244,2026-01-19,190,4,-10.0,2,2,-7.929825,CHN,TWN,None,...,4,39.9289,116.3880,4,39.9289,116.3880,4,25.0478,121.5320,https://foreignpolicy.com/2026/01/19/china-taiwan-invasion-failed-xi-disaster/?tpcc=recirc_trending062921
3245,2026-01-19,190,4,-10.0,6,6,-7.929825,CHN,TWN,None,...,4,39.9289,116.3880,4,25.0478,121.5320,4,25.0478,121.5320,https://foreignpolicy.com/2026/01/19/china-taiwan-invasion-failed-xi-disaster/?tpcc=recirc_trending062921
8340,2026-01-04,150,4,-7.2,16,16,-5.497333,CHN,TWN,MIL,...,3,38.8951,-77.0364,4,35.6850,139.7510,3,38.8951,-77.0364,https://news.webindia123.com/news/Articles/World/20260104/4401029.html
8343,2026-01-04,150,4,-7.2,8,8,-5.497333,CHN,TWN,MIL,...,4,25.0478,121.5320,3,38.8951,-77.0364,3,38.8951,-77.0364,https://news.webindia123.com/news/Articles/World/20260104/4401029.html
8344,2026-01-04,150,4,-7.2,16,16,-5.497333,CHN,TWN,MIL,...,3,38.8951,-77.0364,3,38.8951,-77.0364,3,38.8951,-77.0364,https://news.webindia123.com/news/Articles/World/20260104/4401029.html
3023,2025-12-31,150,4,-7.2,12,12,-4.138128,TWN,CHN,None,...,4,25.0478,121.5320,4,39.9289,116.3880,4,25.0478,121.5320,https://www.prokerala.com/news/articles/a1713359.html


In [18]:
# 중복된 위치/시간/사건을 하나로 합치고 보도량은 더하기
mil_df_cleaned = mil_df.groupby(
    ['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long'], 
    as_index=False
).agg({
    'NumArticles': 'sum',
    'NumMentions': 'sum',
    'AvgTone': 'mean',
    'GoldsteinScale': 'mean',
    'SOURCEURL': 'first' # URL은 하나만 남김
})

print(f"클리닝 후 데이터 수: {len(mil_df_cleaned)}")

클리닝 후 데이터 수: 1506


In [23]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

geolocator = Nominatim(user_agent="gdelt_roi_analysis")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

def reverse_geocode(lat, lon):
    empty = {'country': '', 'country_code': '', 'state': '', 'city': ''}
    try:
        location = reverse(f"{lat}, {lon}", language='en')
        if not location:
            return empty
        addr = location.raw.get('address', {})
        return {
            'country': addr.get('country', ''),
            'country_code': addr.get('country_code', '').upper(),
            'state': addr.get('state', ''),
            'city': addr.get('city') or addr.get('town') or addr.get('county', ''),
        }
    except:
        return empty

# 중복 좌표 제거 후 역지오코딩 (API 호출 최소화)
unique_coords = mil_df_cleaned[['ActionGeo_Lat', 'ActionGeo_Long']].drop_duplicates()
print(f"고유 좌표 수: {len(unique_coords)}")  # 실제 API 호출 횟수

results = []
for _, row in unique_coords.iterrows():
    result = reverse_geocode(row['ActionGeo_Lat'], row['ActionGeo_Long'])
    result['ActionGeo_Lat'] = row['ActionGeo_Lat']
    result['ActionGeo_Long'] = row['ActionGeo_Long']
    results.append(result)

geo_df = pd.DataFrame(results)

# 원본에 합치기
mil_df_cleaned = mil_df_cleaned.merge(geo_df, on=['ActionGeo_Lat', 'ActionGeo_Long'], how='left')

고유 좌표 수: 170


In [29]:
# 국가별 분포 확인
print(mil_df_cleaned['country_code'].value_counts())

country_code
CN    728
TW    520
      170
US     36
JP     20
IN      5
PH      3
GB      3
TR      2
SO      2
CA      2
UA      2
TH      1
IR      1
IT      1
VN      1
KP      1
KH      1
LY      1
SG      1
LT      1
AU      1
ES      1
IL      1
NZ      1
Name: count, dtype: int64


In [26]:
# 1. 공백 170건 좌표 확인
empty_coords = mil_df_cleaned[mil_df_cleaned['country_code'] == ''][
    ['ActionGeo_Lat', 'ActionGeo_Long']
].drop_duplicates()

print("=== 공백 좌표 목록 ===")
print(empty_coords)

# 대만해협 범위 체크 (대략적인 범위)
# 위도: 22~27, 경도: 118~123
taiwan_strait = empty_coords[
    (empty_coords['ActionGeo_Lat'].between(22, 27)) &
    (empty_coords['ActionGeo_Long'].between(118, 123))
]
print(f"\n대만해협 범위 내 좌표: {len(taiwan_strait)}건")
print(taiwan_strait)

=== 공백 좌표 목록 ===
      ActionGeo_Lat  ActionGeo_Long
9           15.0000        115.0000
15          29.0000        125.0000
32          24.0000        119.0000
150         21.4167        121.5000
588         20.0000        135.0000
711         36.0000        124.0000
1363        31.4167         34.3333
1461        25.0000        -90.0000

대만해협 범위 내 좌표: 1건
    ActionGeo_Lat  ActionGeo_Long
32           24.0           119.0


In [30]:
# 현재 컬럼 확인
print(mil_df_cleaned.columns.tolist())

['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long', 'NumArticles', 'NumMentions', 'AvgTone', 'GoldsteinScale', 'SOURCEURL', 'country', 'country_code', 'state', 'city']


In [31]:
print(mil_df.columns.tolist())

['SQLDATE', 'EventCode', 'QuadClass', 'GoldsteinScale', 'NumMentions', 'NumArticles', 'AvgTone', 'Actor1CountryCode', 'Actor2CountryCode', 'Actor1Type1Code', 'Actor2Type1Code', 'Actor1Geo_Type', 'Actor1Geo_Lat', 'Actor1Geo_Long', 'Actor2Geo_Type', 'Actor2Geo_Lat', 'Actor2Geo_Long', 'ActionGeo_Type', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL']


In [32]:
# mil_df 원본에서 바로 확인
# 역지오코딩 결과(country_code)는 mil_df_cleaned에 있으니 merge 필요

cn_cases = mil_df_cleaned[mil_df_cleaned['country_code'] == 'CN']

print("=== state 분포 ===")
print(cn_cases['state'].value_counts().head(15))

print("\n=== city 분포 ===")
print(cn_cases['city'].value_counts().head(15))

=== state 분포 ===
state
             422
Guangdong    147
Fujian        32
Liaoning      26
Jiangsu       22
Zhejiang      13
Guizhou        9
Shaanxi        7
Jilin          6
Jiangxi        6
Henan          6
Sichuan        5
Shanghai       5
Hunan          4
Hainan         3
Name: count, dtype: int64

=== city 분포 ===
city
Xicheng District                  420
Qingyuan City                     127
Dong'anshan Subdistrict            18
                                   16
Xuyi County                        13
Zhanjiang City                     11
Shuicheng                           8
Xiamen                              7
Xinchang County                     5
Nanyang                             5
Yan'an                              5
Shanghai                            5
Jingcheng                           5
Dongcheng                           5
Jianghua Yao Autonomous County      4
Name: count, dtype: int64


In [33]:
# mil_df_cleaned 생성 코드 확인 필요
# Actor2Geo 컬럼을 merge로 다시 붙이는 방법도 있음

mil_df_cleaned = mil_df_cleaned.merge(
    mil_df[['ActionGeo_Lat', 'ActionGeo_Long', 
            'Actor2Geo_Lat', 'Actor2Geo_Long']].drop_duplicates(),
    on=['ActionGeo_Lat', 'ActionGeo_Long'],
    how='left'
)

print(mil_df_cleaned.columns.tolist())

['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long', 'NumArticles', 'NumMentions', 'AvgTone', 'GoldsteinScale', 'SOURCEURL', 'country', 'country_code', 'state', 'city', 'Actor2Geo_Lat', 'Actor2Geo_Long']


In [34]:
# Xicheng District 420건의 Actor2Geo 역지오코딩
xicheng = mil_df_cleaned[mil_df_cleaned['city'] == 'Xicheng District'][
    ['Actor2Geo_Lat', 'Actor2Geo_Long']
].drop_duplicates()

print(f"고유 Actor2Geo 좌표 수: {len(xicheng)}")

고유 Actor2Geo 좌표 수: 22


In [35]:
results3 = []
for _, row in xicheng.iterrows():
    result = reverse_geocode(row['Actor2Geo_Lat'], row['Actor2Geo_Long'])
    result['Actor2Geo_Lat'] = row['Actor2Geo_Lat']
    result['Actor2Geo_Long'] = row['Actor2Geo_Long']
    results3.append(result)

actor2_xicheng_df = pd.DataFrame(results3)
print(actor2_xicheng_df[['country_code', 'state', 'city', 'Actor2Geo_Lat', 'Actor2Geo_Long']])

   country_code                 state                     city  Actor2Geo_Lat  \
0            CN                               Xicheng District      39.928900   
1            TW                                         Taipei      25.047800   
2            TW                              Jincheng Township      24.436700   
3            US                Kansas             Smith County      39.828175   
4            CN                             Dongcheng District      39.904400   
5            TW                                  Puli Township      24.000000   
6            TW                                         Chiayi      23.479200   
7                                                                    6.339870   
8                                                                   15.000000   
9            TW                                   Taoyuan City      25.032700   
10           JP                                        Tatsuno      36.000000   
11                          

In [39]:
mil_df_cleaned.sort_values(by=['SQLDATE'], ascending=False).head(30)

,SQLDATE,EventCode,ActionGeo_Lat,ActionGeo_Long,NumArticles,NumMentions,AvgTone,GoldsteinScale,SOURCEURL,country,country_code,state,city,Actor2Geo_Lat,Actor2Geo_Long
22363,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,41.000000,123.0000
22352,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,36.000000,138.0000
22342,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,39.928900,116.3880
22343,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,25.047800,121.5320
22344,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,24.436700,118.3180
22345,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,39.828175,-98.5795
22346,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,39.904400,116.3910
22347,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,24.000000,121.0000
22349,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,6.339870,-162.6750
22350,2026-04-30,150,39.9289,116.388,9,9,0.559158,-7.2,https://focustaiwan.tw/politics/202604300010,China,CN,,Xicheng District,15.000000,115.0000


In [40]:
# 원본 mil_df 확인
print(f"mil_df 행 수: {len(mil_df)}")
print(f"mil_df_cleaned 행 수: {len(mil_df_cleaned)}")

# 원본에서 같은 패턴이 있는지 확인
# 같은 SOURCEURL에 행이 몇 개씩 있는지
url_counts = mil_df.groupby('SOURCEURL').size()
print(f"\n원본 URL당 행 수 분포:")
print(url_counts.value_counts().head(10))

mil_df 행 수: 2049
mil_df_cleaned 행 수: 22364

원본 URL당 행 수 분포:
1     1008
2      337
3       75
4       16
5        6
6        3
7        1
14       1
9        1
Name: count, dtype: int64
